# Chunk Diagnostics
How much did meta-chunking actually kick in vs. passing turns through as-is?

**Three categories per answer turn:**
- **pass-through**: < 10 sentences → meta-chunking never ran, chunk = full turn
- **eligible, unsplit**: ≥ 10 sentences, but meta-chunking found no boundaries → still 1 chunk
- **split**: ≥ 10 sentences, meta-chunking produced > 1 chunk

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("..")
CHUNKS_DIR = ROOT / "data" / "csv" / "chunks"
BANKS = ["BoE", "ECB", "Fed"]

In [2]:
def diagnose(bank):
    df = pd.read_csv(CHUNKS_DIR / f"{bank}_chunks.csv")
    answers = df[df["turn_type"] == "answer"].copy()

    # Aggregate to turn level
    turns = (
        answers.groupby(["doc_id", "turn_idx"])
        .agg(total_sents=("n_sentences", "sum"), n_chunks=("chunk_id", "count"))
        .reset_index()
    )

    turns["category"] = np.where(
        turns["total_sents"] < 10, "pass-through",
        np.where(turns["n_chunks"] == 1, "eligible, unsplit", "split")
    )

    n = len(turns)
    cat = turns["category"].value_counts().reindex(["pass-through", "eligible, unsplit", "split"], fill_value=0)

    print(f"\n{'='*46}")
    print(f"  {bank}  -  {n} answer turns")
    print(f"{'='*46}")
    print(f"  {'category':<22} {'N':>6}  {'%':>6}")
    print(f"  {'-'*38}")
    for label, count in cat.items():
        print(f"  {label:<22} {count:>6}  {100*count/n:>5.1f}%")

    eff_full = cat["pass-through"] + cat["eligible, unsplit"]
    print(f"  {'-'*38}")
    print(f"  {'effectively full turns':<22} {eff_full:>6}  {100*eff_full/n:>5.1f}%")

    split_turns = turns[turns["category"] == "split"]
    if len(split_turns):
        print(f"\n  Split turns - chunks per turn:")
        print(f"    mean   {split_turns['n_chunks'].mean():.2f}")
        print(f"    median {split_turns['n_chunks'].median():.1f}")
        print(f"    p75    {split_turns['n_chunks'].quantile(0.75):.1f}")
        print(f"    max    {split_turns['n_chunks'].max()}")

    print(f"\n  Sentence count distribution (all answer turns):")
    for p, v in [(25, turns['total_sents'].quantile(0.25)),
                 (50, turns['total_sents'].median()),
                 (75, turns['total_sents'].quantile(0.75)),
                 (90, turns['total_sents'].quantile(0.90))]:
        print(f"    p{p:<3}  {v:.1f} sents")
    print(f"    max   {turns['total_sents'].max()} sents")

    return turns

results = {bank: diagnose(bank) for bank in BANKS}


  BoE  -  736 answer turns
  category                    N       %
  --------------------------------------
  pass-through              467   63.5%
  eligible, unsplit         266   36.1%
  split                       3    0.4%
  --------------------------------------
  effectively full turns    733   99.6%

  Split turns - chunks per turn:
    mean   2.00
    median 2.0
    p75    2.0
    max    2

  Sentence count distribution (all answer turns):
    p25   4.0 sents
    p50   8.0 sents
    p75   11.0 sents
    p90   15.0 sents
    max   33 sents

  ECB  -  2445 answer turns
  category                    N       %
  --------------------------------------
  pass-through             2317   94.8%
  eligible, unsplit         124    5.1%
  split                       4    0.2%
  --------------------------------------
  effectively full turns   2441   99.8%

  Split turns - chunks per turn:
    mean   2.00
    median 2.0
    p75    2.0
    max    2

  Sentence count distribution (all answe

In [ ]:
def summary_table(threshold=10):
    rows = []
    for bank in BANKS:
        df = pd.read_csv(CHUNKS_DIR / f"{bank}_chunks.csv")
        answers = df[df["turn_type"] == "answer"].copy()
        turns = (
            answers.groupby(["doc_id", "turn_idx"])
            .agg(total_sents=("n_sentences", "sum"), n_chunks=("chunk_id", "count"))
            .reset_index()
        )
        n = len(turns)
        pt  = (turns["total_sents"] < threshold).sum()
        spl = (turns["n_chunks"] > 1).sum()
        elig_unsplit = n - pt - spl
        eff = pt + elig_unsplit
        rows.append([bank, f"{100*pt/n:.1f}%", f"{100*elig_unsplit/n:.1f}%", f"{100*spl/n:.1f}%", f"{100*eff/n:.1f}%"])

    cols = ["Bank", f"Pass-through (<{threshold} sents)", "Eligible, unsplit", "Actually split", "Effectively full turns"]
    tbl = pd.DataFrame(rows, columns=cols).set_index("Bank")
    print(f"\nMeta-chunking almost never split answer turns  [threshold={threshold}]")
    print(tbl.to_string())

summary_table(10)
summary_table(8)
summary_table(5)

## Counterfactual: lower eligibility thresholds\n\nWhat if we lowered the sentence threshold below which meta-chunking runs? Currently it's 10. Below we simulate thresholds of 5 and 8 — showing how many more turns would become eligible and what fraction would still end up unsplit (i.e. chunking ran but found no boundaries).\n\nNote: this is a **lower-bound estimate** of splits. With a lower threshold, the actual meta-chunking signal may be noisier on shorter turns, so real split counts could differ.

In [ ]:
def diagnose_threshold(bank, threshold):
    df = pd.read_csv(CHUNKS_DIR / f"{bank}_chunks.csv")
    answers = df[df["turn_type"] == "answer"].copy()

    turns = (
        answers.groupby(["doc_id", "turn_idx"])
        .agg(total_sents=("n_sentences", "sum"), n_chunks=("chunk_id", "count"))
        .reset_index()
    )

    # With a lower threshold, turns previously below 10 would now be eligible.
    # We can't know if they'd split (no re-running embeddings), so we count them
    # as "newly eligible" and show how many currently-split turns we'd add at minimum.
    n = len(turns)
    below_old   = (turns["total_sents"] < 10).sum()
    below_new   = (turns["total_sents"] < threshold).sum()
    newly_eligible = below_old - below_new   # turns that would become eligible

    # Of the already-eligible turns (>= 10 sents), keep split count as-is
    already_split = (turns["n_chunks"] > 1).sum()

    print(f"  {bank:<6}  threshold={threshold}:")
    print(f"    pass-through (< {threshold} sents):   {below_new:>5}  ({100*below_new/n:.1f}%)")
    print(f"    newly eligible vs threshold=10: +{newly_eligible} turns")
    print(f"    currently split (>= 10 sents):  {already_split:>5}  ({100*already_split/n:.1f}%)")
    print()

print("Counterfactual: how many turns become eligible at lower thresholds")
print("(split count shown is only for turns already >= 10 sents)\n")
for threshold in [5, 8]:
    print(f"--- threshold = {threshold} " + "-"*30)
    for bank in BANKS:
        diagnose_threshold(bank, threshold)


In [ ]:
for threshold in [5, 8]:
    rows = []
    for bank in BANKS:
        df = pd.read_csv(CHUNKS_DIR / f"{bank}_chunks.csv")
        answers = df[df["turn_type"] == "answer"].copy()
        turns = (
            answers.groupby(["doc_id", "turn_idx"])
            .agg(total_sents=("n_sentences", "sum"), n_chunks=("chunk_id", "count"))
            .reset_index()
        )
        n = len(turns)
        pt  = (turns["total_sents"] < threshold).sum()
        spl = (turns["n_chunks"] > 1).sum()
        elig_unsplit = n - pt - spl
        eff = pt + elig_unsplit
        rows.append({
            "Bank": bank,
            f"Pass-through (<{threshold} sents)": f"{100*pt/n:.1f}%",
            "Eligible, unsplit":                   f"{100*elig_unsplit/n:.1f}%",
            "Actually split":                      f"{100*spl/n:.1f}%",
            "Effectively full turns":              f"{100*eff/n:.1f}%",
        })

    tbl = pd.DataFrame(rows).set_index("Bank")
    display(
        tbl.style
        .set_caption(f"Counterfactual: threshold = {threshold} sentences")
        .set_table_styles([
            {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "14px"), ("padding-bottom", "8px")]},
            {"selector": "th", "props": [("background-color", "#2d2d2d"), ("color", "white"), ("text-align", "center"), ("padding", "6px 12px")]},
            {"selector": "td", "props": [("text-align", "center"), ("padding", "5px 12px")]},
            {"selector": "tr:nth-child(even)", "props": [("background-color", "#1e1e1e")]},
        ])
        .applymap(lambda v: "font-weight: bold; color: #7ec8a0" if float(v.rstrip("%")) > 95 else "", subset=["Effectively full turns"])
    )